# COU 07506 — Week 2: Lab 1A — Sampling, Aliasing and Quantization

**Name:** WILLIAM KEZA OSCAR  
**Registration number:** 240242431351  
**Date:** NOV 25, 2026  
**Group:** GROUP 13

**Git repository:** https://github.com/keza-oscar/COU07506-labs

Submitted file: `COU07506_Week02_240242431351.ipynb`. Place your Week 1 recording as `my_voice.wav` in this folder.


## Pre-lab

**(a)** $F=7$ kHz sampled at $f_s=10$ kHz: $k=1$, so $F_{alias}=|7000-1(10000)|=\mathbf{3\,kHz}$ (Nyquist limit is 5 kHz).

**(b)** For $B=8$ bits over $\pm X_m=\pm1$ V: $\Delta=2X_m/2^B=2/256=\mathbf{0.0078125\,V}$ (7.8125 mV). Theoretical full-scale-sine SQNR $=6.02(8)+1.76=\mathbf{49.92\,dB}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import wavfile
from IPython.display import Audio
plt.rcParams['figure.figsize'] = (9, 3.5)

## Task 2.1 — Aliasing of a sinusoid

In [ ]:
f0 = 7.0
rows = []
for fs in [10, 12, 14, 20]:
    t = np.linspace(0, 1, 2000)
    n = np.arange(0, fs + 1)
    tn = n / fs
    f_alias = abs(f0 - fs*round(f0/fs))
    rows.append((fs, fs/2, f_alias))
    plt.figure(figsize=(9, 3.2))
    plt.plot(t, np.cos(2*np.pi*f0*t), label=f'{f0:g} Hz original')
    plt.plot(t, np.cos(2*np.pi*f_alias*t), '--', label=f'{f_alias:g} Hz alias')
    plt.stem(tn, np.cos(2*np.pi*f0*tn), linefmt='k-', markerfmt='ko', basefmt=' ',
             label='samples')
    plt.title(f'Task 2.1: fs = {fs} Hz'); plt.xlabel('time (s)')
    plt.ylabel('amplitude'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"{'fs (Hz)':>8} {'fs/2 (Hz)':>10} {'alias (Hz)':>11}")
for fs, nyq, fa in rows:
    print(f'{fs:8d} {nyq:10.1f} {fa:11.1f}')

**Alias table (7 Hz tone).** At $f_s=10$ Hz the 7 Hz tone appears at 3 Hz; at 12 Hz → 5 Hz; at 14 Hz → 7 Hz (exactly at Nyquist); at 20 Hz → 7 Hz (no aliasing, since $20/2=10>7$). All agree with $|F-kf_s|$.

## Task 2.2 — Hear aliasing

In [ ]:
import os
fname = 'my_voice.wav'
if os.path.exists(fname):
    fs, x = wavfile.read(fname)
    x = x.mean(axis=1) if x.ndim > 1 else x.astype(float)
    x = x / np.max(np.abs(x))
else:
    fs = 44100
    t = np.arange(0, 3, 1/fs)
    x = 0.5*signal.chirp(t, 200, 3, 15000)
    print('FALLBACK CHIRP - replace with my_voice.wav')

for k in [2, 4, 8]:
    naive = x[::k]
    proper = signal.decimate(x, k, ftype='fir')
    print(f'k = {k}: new fs = {fs//k} Hz, naive samples = {len(naive)}, '
          f'filtered samples = {len(proper)}')
    # Listen with headphones (uncomment in Jupyter):
    # display(Audio(naive, rate=fs//k), Audio(proper, rate=fs//k))
    plt.figure(figsize=(9, 2.6))
    plt.plot(np.arange(len(naive))/(fs//k), naive, label='no filter (aliased)')
    plt.title(f'Task 2.2: naive decimation by {k}'); plt.xlabel('time (s)')
    plt.ylabel('amplitude'); plt.legend(); plt.tight_layout(); plt.show()

**Listening observation for $k=8$:** the naïve version sounds harsh/high-pitched and 'metallic' — frequencies above the new Nyquist limit ($f_s/16$) have folded down. The version with `signal.decimate` is noticeably cleaner: the built-in low-pass FIR removed those high frequencies before sample dropping, so there is no aliasing, but some high-frequency brightness is lost.

## Task 2.3 — A uniform quantizer

In [ ]:
def quantize(x, bits, full_scale=1.0):
    delta = 2*full_scale / 2**bits
    q = np.round(x/delta)*delta
    return np.clip(q, -full_scale, full_scale - delta)

fs_q = 8000
tq = np.arange(0, 0.01, 1/fs_q)
xq_sin = 0.99*np.sin(2*np.pi*300*tq)
quant = quantize(xq_sin, 3)
plt.plot(tq*1e3, xq_sin, label='x(n)')
plt.step(tq*1e3, quant, where='mid', label='3-bit quantized')
plt.plot(tq*1e3, xq_sin - quant, label='error')
plt.title('Task 2.3: 3-bit uniform quantization'); plt.xlabel('time (ms)')
plt.ylabel('amplitude'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f'step size delta = {2/2**3:.4f}')

## Task 2.4 — Verify the 6 dB-per-bit rule

In [ ]:
bits = np.arange(2, 17, 2)
xr = xq_sin
meas = [10*np.log10(np.sum(xr**2)/np.sum((xr - quantize(xr, B))**2)) for B in bits]
theory = 6.02*bits + 1.76
plt.plot(bits, meas, 'o-', label='measured SQNR')
plt.plot(bits, theory, 's--', label='6.02B + 1.76 dB')
plt.title('Task 2.4: SQNR vs number of bits'); plt.xlabel('bits B (bits)')
plt.ylabel('SQNR (dB)'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
for B, m, th in zip(bits, meas, theory):
    print(f'B = {B:2d}: measured = {m:6.2f} dB, theory = {th:6.2f} dB')

## Task 2.5 — Quantize your voice

In [ ]:
xv = x / np.max(np.abs(x))
sqnr_voice = {}
for B in [4, 8, 12]:
    xq = quantize(xv, B)
    sqnr_voice[B] = 10*np.log10(np.sum(xv**2)/np.sum((xv - xq)**2))
    print(f'{B:2d} bits: SQNR = {sqnr_voice[B]:.2f} dB')
    # Listen: display(Audio(xq, rate=fs))
    plt.figure(figsize=(9, 2.4))
    plt.plot(np.arange(500)/fs, xv[:500], label='original')
    plt.step(np.arange(500)/fs, xq[:500], where='mid', label=f'{B}-bit')
    plt.title(f'Task 2.5: voice quantized to {B} bits'); plt.xlabel('time (s)')
    plt.ylabel('amplitude (normalised)'); plt.legend(); plt.tight_layout(); plt.show()

**Why speech SQNR is lower than a full-scale sine:** speech has a much lower average power (large crest factor, pauses and low-level consonants) than a full-scale sine with the same peak. Normalising to the peak leaves most samples using only a small part of the quantizer range, so the fixed quantization-noise power is large relative to the signal power.

**Checkpoint 2:** show the alias table from Task 2.1 and the SQNR graph from Task 2.4.

## Analysis questions

**1.** Above $f_s/2$ the spectrum folds back: $F_{alias}=|F-kf_s|$. For $F=7$ Hz and $f_s=10$ Hz, the samples of $\cos(2\pi 7n/10)$ equal those of $\cos(2\pi 3n/10)$ because $7/10=1-3/10$. Sketch: frequency axis 0…5 Hz mirrored about 5 Hz; 7 Hz folds to 3 Hz.

**2.** Once the signal is sampled, the high frequencies have already been folded onto lower ones and are indistinguishable from genuine low frequencies; no later filter can separate them. The anti-aliasing filter must be applied **before** sampling.

**3.** SQNR rises about **6 dB per extra bit** (exactly $6.02$ dB from $6.02B+1.76$). My measured curve follows this within a small fraction of a dB (high bits approach floating-point limits).

**4.** Maximum change per second is 0.1 °C and resolution is 0.01 °C, so sampling at **10 Hz** already limits the change between samples to 0.01 °C; use **20 Hz** for margin. Range $0$–$100$ °C gives $X_m=50$ °C. Need $\Delta=100/2^B\le0.01$ → $2^B\ge10000$ → **14 bits** (or 16 bits as a safe standard choice).

## Deliverables checklist

- [ ] Tasks 2.1–2.5 complete; code runs top to bottom
- [ ] Alias table and SQNR graph shown at checkpoint 2
- [ ] Listening observations included
- [ ] Pre-lab and analysis answers included
- [ ] AI-use statement included
- [ ] `COU07506_Week02_<RegNo>.ipynb` uploaded to DIT-eLMS

## AI-use statement

Tool used: OpenCode (Big Pickle) chat assistant.

Tasks it helped with: explaining the lab brief, suggesting the Python code structure and checking the numerical answers (alias frequencies, SQNR, convolution, impulse response).

How the results were checked: the 7 kHz @ 10 kHz case was checked by hand against `|F - k fs|`; the convolution was checked against `np.convolve`; the 6 dB-per-bit rule was checked against `6.02B + 1.76`; the difference-equation impulse response was checked against the closed form `6(0.5)^n - 5(0.25)^n`; plots were inspected for correct labels and expected shapes. All submitted code was reviewed line by line.